# Batch XML De-Identification — Multiple Files

Runs the full two-pass de-identification pipeline across all XML files in `analysis/`.

- **Pass 1** — Structural XML rules (patient name, DOB, SSN, dates, ZIP, phone, street)
- **Pass 2** — ZeroShot NER on every text node

Output:
- `analysis/deid/<filename>` — de-identified XML per file
- `analysis/Batch_DeID_Results.xlsx` — one sheet per file + summary sheet


In [ ]:
import json, os, re, copy, glob
import xml.etree.ElementTree as ET
from datetime import datetime, date as date_type
import pandas as pd

os.environ["JAVA_HOME"]             = "/opt/homebrew/opt/openjdk@11"
os.environ["PYSPARK_PYTHON"]        = __import__("sys").executable
os.environ["PYSPARK_DRIVER_PYTHON"] = __import__("sys").executable

with open("spark_jsl.json") as f:
    license_keys = json.load(f)
locals().update(license_keys)
os.environ.update(license_keys)

INPUT_DIR  = "analysis"
OUTPUT_DIR = "analysis/deid"
EXCEL_OUT  = "analysis/Batch_DeID_Results.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Patient ID lookup — all patients found across the 10 files ────────────
PATIENT_ID_MAP = {
    "Myra Jones"       : "PT-00001",
    "Bryce Zemlak"     : "PT-00002",
    "Elizabeth Itasca" : "PT-00003",
    "Kimberly Olympic" : "PT-00004",
    "Grant Custer"     : "PT-00005",
    "Minh Kulas"       : "PT-00006",
}

def get_patient_id(name: str) -> str:
    return PATIENT_ID_MAP.get(name.strip(), f"PT-{abs(hash(name)) % 90000 + 10000}")

print("Setup done.")
print("Patient map:", PATIENT_ID_MAP)


In [ ]:
# Release any stale license lock before starting
try:
    from pyspark.sql import SparkSession
    existing = SparkSession.getActiveSession()
    if existing:
        print("Stopping existing Spark session...")
        existing.stop()
        import time; time.sleep(3)
except Exception as e:
    print(f"No existing session: {e}")

import sparknlp, sparknlp_jsl
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline

# Re-import standard library after wildcard
import re, copy, json, os, glob
from datetime import datetime, date as date_type

spark = sparknlp_jsl.start(license_keys["SECRET"])
print(f"Spark NLP {sparknlp.version()} / JSL {sparknlp_jsl.version()}")


## Helper Functions

In [ ]:
_DATE_FORMATS = [
    "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d", "%m-%d-%Y", "%d-%m-%Y",
    "%d.%m.%Y", "%m.%d.%Y", "%Y/%m/%d",
    "%B %d, %Y", "%b %d, %Y", "%d %B %Y", "%d %b %Y",
    "%B %d %Y",  "%b %d %Y",
]

def _parse_date(s):
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s.strip())
    for fmt in _DATE_FORMATS:
        try: return datetime.strptime(s, fmt)
        except ValueError: pass
    return None

def yyyymmdd_to_month_year(v):
    try:    return datetime.strptime(v[:8], "%Y%m%d").strftime("%B %Y")
    except Exception: pass
    m = re.search(r'\b(19|20)\d{2}\b', v)
    return m.group(0) if m else "[DATE]"

def yyyymmdd_to_age(v):
    try:
        dt = datetime.strptime(v[:8], "%Y%m%d")
        today = date_type.today()
        age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
        if 0 <= age <= 120: return f"{age} years old"
    except Exception: pass
    return "[DOB]"

def partial_zip(z):
    z = z.strip()
    return z[:3] + "X" * (len(z) - 3) if len(z) > 3 else z

def strip_preamble(raw):
    """Remove non-XML preamble lines (e.g. browser info before the root tag)."""
    if raw.lstrip().startswith("<"):
        return raw
    return re.sub(r'^.*?(?=<)', '', raw, flags=re.DOTALL)

print("Helpers ready.")


## Build ZeroShot NER Pipeline (once for all files)

In [ ]:
documentAssembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
splitter = (InternalDocumentSplitter()
    .setInputCols("document").setOutputCol("sentence")
    .setSplitMode("recursive").setSplitPatterns([r"\s+|(?<=\G.{512})"])
    .setPatternsAreRegex(True).setChunkSize(512).setChunkOverlap(50)
    .setEnableSentenceIncrement(True))
tokenizer     = Tokenizer().setInputCols("sentence").setOutputCol("token")
tokenizer_doc = Tokenizer().setInputCols("document").setOutputCol("token_doc")

labels = ["DOCTOR","PATIENT","DATE_OF_BIRTH","DATE","CITY","STREET","STATE",
          "COUNTRY","PHONE","EMAIL","ZIP","USERNAME","ID","BIOID",
          "ORGANIZATION","MEDICAL_RECORD_NUMBER","SSN","AGE"]

zero_shot_ner = (PretrainedZeroShotNERChunker
    .pretrained("zeroshot_ner_deid_subentity_docwise_medium","en","clinical/models")
    .setInputCols("sentence").setOutputCol("ner_zero_shot")
    .setPredictionThreshold(0.7).setLabels(labels).setBatchSize(8))

zip_parser      = (ContextualParserModel.pretrained("zip_parser","en","clinical/models")
                   .setInputCols(["document","token_doc"]).setOutputCol("zip_chunks"))
dob_parser      = (ContextualParserModel.pretrained("date_of_birth_parser","en","clinical/models")
                   .setInputCols(["document","token_doc"]).setOutputCol("dob_chunks"))
email_matcher   = (RegexMatcherInternalModel.pretrained("email_matcher","en","clinical/models")
                   .setInputCols(["document"]).setOutputCol("email_chunks"))
country_matcher = (TextMatcherInternalModel.pretrained("country_matcher","en","clinical/models")
                   .setInputCols(["document","token_doc"]).setOutputCol("country_chunks")
                   .setMergeOverlapping(True))

chunk_merge_ner = (ChunkMergeModel().setInputCols("ner_zero_shot").setOutputCol("ner_merged")
    .setMergeOverlapping(True).setSelectionStrategy("DiverseLonger").setResetSentenceIndices(True)
    .setReplaceDict({"DATE_OF_BIRTH":"DOB","MEDICAL_RECORD_NUMBER":"MEDICALRECORD"}))
chunk_merge_rules = (ChunkMergeModel()
    .setInputCols("zip_chunks","email_chunks","dob_chunks","country_chunks")
    .setOutputCol("rules_merged").setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))
chunk_merge_final = (ChunkMergeModel()
    .setInputCols("ner_merged","rules_merged").setOutputCol("ner_chunk")
    .setMergeOverlapping(True).setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"]).setSelectionStrategy("Sequential"))

nlp_pipeline = Pipeline(stages=[documentAssembler,splitter,tokenizer,tokenizer_doc,
    zero_shot_ner,chunk_merge_ner,zip_parser,dob_parser,email_matcher,country_matcher,
    chunk_merge_rules,chunk_merge_final])

nlp_model = nlp_pipeline.fit(spark.createDataFrame([[""]],("text",)))
print("NLP pipeline ready.")


In [ ]:
@F.udf(StringType())
def custom_deid_udf(text, begins, ends, results, meta_list):
    import re
    from datetime import datetime, date as date_type

    _PATIENT_ID_MAP = {
        "Myra Jones": "PT-00001", "Bryce Zemlak": "PT-00002",
        "Elizabeth Itasca": "PT-00003", "Kimberly Olympic": "PT-00004",
        "Grant Custer": "PT-00005", "Minh Kulas": "PT-00006",
    }
    _DATE_FORMATS = [
        "%m/%d/%Y","%d/%m/%Y","%Y-%m-%d","%m-%d-%Y","%d-%m-%Y",
        "%d.%m.%Y","%m.%d.%Y","%Y/%m/%d",
        "%B %d, %Y","%b %d, %Y","%d %B %Y","%d %b %Y","%B %d %Y","%b %d %Y",
    ]
    def _parse_date(s):
        s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', s.strip())
        for fmt in _DATE_FORMATS:
            try: return datetime.strptime(s, fmt)
            except ValueError: pass
        return None
    def _get_patient_id(name):
        return _PATIENT_ID_MAP.get(name, f"PT-{abs(hash(name)) % 90000 + 10000}")
    def _date_to_month_year(s):
        dt = _parse_date(s)
        if dt: return dt.strftime("%B %Y")
        m = re.search(r'\b(19|20)\d{2}\b', s)
        return m.group(0) if m else "[DATE]"
    def _dob_to_age(s):
        dt = _parse_date(s)
        if dt:
            today = date_type.today()
            age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
            if 0 <= age <= 120: return f"{age} years old"
        return "[DOB]"
    def _partial_zip(z):
        z = z.strip()
        return z[:3] + "X" * (len(z) - 3) if len(z) > 3 else z

    if not text or not results: return text
    chunks = []
    for i, chunk_text in enumerate(results):
        b = begins[i] if begins else 0
        e = ends[i]   if ends   else 0
        meta   = meta_list[i] if meta_list else {}
        entity = meta.get("entity","") if meta else ""
        chunks.append((b, e, entity, chunk_text))
    chunks.sort(key=lambda x: x[0], reverse=True)
    result = text
    for begin, end, entity, chunk_text in chunks:
        if   entity == "DOCTOR":          continue
        elif entity == "PATIENT":         replacement = _get_patient_id(chunk_text)
        elif entity == "DATE":            replacement = _date_to_month_year(chunk_text)
        elif entity == "DOB":             replacement = _dob_to_age(chunk_text)
        elif entity in ("ZIP","ZIPCODE"): replacement = _partial_zip(chunk_text)
        else:                             replacement = f"[{entity}]"
        result = result[:begin] + replacement + result[end + 1:]
    return result

print("UDF registered.")


## Pass 1 — Structural XML Rules (per file)

In [ ]:
SSN_ROOT = "2.16.840.1.113883.4.1"

def pass1_structural(root, ns, changelog):
    """Apply structural XML de-ID rules. Returns number of changes."""
    changes = 0
    def t(name): return f"{{{ns}}}{name}" if ns else name
    def log(loc, ftype, orig, rule, deid):
        changelog.append({"XML Location":loc,"Field Type":ftype,
                          "Original Value":orig,"Rule Applied":rule,"De-identified Value":deid})

    # 1. Patient name
    for patient_el in root.iter(t("patient")):
        for name_el in patient_el.findall(f".//{t('name')}"):
            given  = " ".join(g.text.strip() for g in name_el.findall(t("given"))  if g.text)
            family = " ".join(f.text.strip() for f in name_el.findall(t("family")) if f.text)
            full   = f"{given} {family}".strip()
            if full:
                pid = get_patient_id(full)
                log("patient/name","PATIENT_NAME",full,"→ Patient ID",pid)
                for g in name_el.findall(t("given")):  g.text = ""
                for f in name_el.findall(t("family")): f.text = pid
                changes += 1

    # 2. Doctor names — log only, keep as-is
    for tag in ("assignedPerson","responsibleParty"):
        for el in root.iter(t(tag)):
            for nm in el.findall(f".//{t('name')}"):
                given  = " ".join(g.text.strip() for g in nm.findall(t("given"))  if g.text)
                family = " ".join(f.text.strip() for f in nm.findall(t("family")) if f.text)
                full   = f"{given} {family}".strip()
                if full: log(f"{tag}/name","DOCTOR_NAME",full,"kept as-is",full)

    # 3. Related persons
    for tag in ("relatedPerson","associatedPerson","informationRecipient"):
        for el in root.iter(t(tag)):
            for nm in el.findall(f".//{t('name')}"):
                given  = " ".join(g.text.strip() for g in nm.findall(t("given"))  if g.text)
                family = " ".join(f.text.strip() for f in nm.findall(t("family")) if f.text)
                full   = f"{given} {family}".strip()
                if full:
                    log(f"{tag}/name","RELATED_PERSON",full,"→ [RELATED_PERSON]","[RELATED_PERSON]")
                    for g in nm.findall(t("given")):  g.text = ""
                    for f in nm.findall(t("family")): f.text = "[RELATED_PERSON]"
                    changes += 1

    # 4. DOB
    for el in root.iter(t("birthTime")):
        v = el.get("value","")
        if v:
            age = yyyymmdd_to_age(v)
            log("birthTime/@value","DOB",v,"→ Age",age)
            el.set("value", age); changes += 1

    # 5. SSN
    for el in root.iter(t("id")):
        if el.get("root") == SSN_ROOT:
            ext = el.get("extension","")
            log("id[@root=SSN]/@extension","SSN",ext,"→ [SSN]","[SSN]")
            el.set("extension","[SSN]"); changes += 1

    # 6. Service dates
    for el in root.iter():
        local = el.tag.split("}")[-1] if "}" in el.tag else el.tag
        if local in {"effectiveTime","low","high","time"}:
            v = el.get("value","")
            if v and re.match(r'^(19|20)\d{6}', v):
                my = yyyymmdd_to_month_year(v)
                log(f"{local}/@value","SERVICE_DATE",v,"→ Month+Year",my)
                el.set("value", my); changes += 1

    # 7. ZIP
    for el in root.iter(t("postalCode")):
        if el.text and el.text.strip():
            z = el.text.strip()
            pz = partial_zip(z)
            log("postalCode","ZIP",z,"→ partial mask",pz)
            el.text = pz; changes += 1

    # 8. Phone
    for el in root.iter(t("telecom")):
        v = el.get("value","")
        if v.startswith("tel:"):
            log("telecom/@value","PHONE",v,"→ tel:[PHONE]","tel:[PHONE]")
            el.set("value","tel:[PHONE]"); changes += 1

    # 9. Street
    for el in root.iter(t("streetAddressLine")):
        if el.text and el.text.strip():
            log("streetAddressLine","STREET",el.text.strip(),"→ [STREET]","[STREET]")
            el.text = "[STREET]"; changes += 1

    return changes

print("Pass 1 function ready.")


## Pass 2 — ZeroShot NER on All Text Nodes (per file)

In [ ]:
def pass2_nlp(root, changelog):
    """Run ZeroShot NER on all text nodes. Returns number of nodes updated."""
    def log(loc, ftype, orig, rule, deid):
        changelog.append({"XML Location":loc,"Field Type":ftype,
                          "Original Value":orig,"Rule Applied":rule,"De-identified Value":deid})

    text_nodes = []
    for el in root.iter():
        if el.text and el.text.strip():
            text_nodes.append((el, "text", el.text))
        if el.tail and el.tail.strip():
            text_nodes.append((el, "tail", el.tail))

    if not text_nodes:
        return 0

    unique_texts = list({txt for _, _, txt in text_nodes})
    df = spark.createDataFrame(list(enumerate(unique_texts)), ["idx","text"])

    result_df = nlp_model.transform(df).withColumn(
        "deid_text",
        custom_deid_udf(F.col("text"), F.col("ner_chunk.begin"),
                        F.col("ner_chunk.end"), F.col("ner_chunk.result"),
                        F.col("ner_chunk.metadata"))
    )

    deid_map = {
        row["text"]: row["deid_text"]
        for row in result_df.select("text","deid_text").collect()
        if row["text"] != row["deid_text"]
    }

    applied = 0
    for el, attr, orig_text in text_nodes:
        deid_text = deid_map.get(orig_text)
        if deid_text:
            tag_name = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            log(f"<{tag_name}> ({attr})","NLP_TEXT",orig_text[:80],"ZeroShot NER",deid_text[:80])
            if attr == "text": el.text = deid_text
            else:              el.tail = deid_text
            applied += 1
    return applied

print("Pass 2 function ready.")


## Batch Processing — All Files

In [ ]:
import copy

input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.txt")))
print(f"Files to process: {len(input_files)}")
print()

# Register CDA namespaces for clean output
ET.register_namespace("",     "urn:hl7-org:v3")
ET.register_namespace("xsi",  "http://www.w3.org/2001/XMLSchema-instance")
ET.register_namespace("sdtc", "urn:hl7-org:sdtc")
ET.register_namespace("cda",  "urn:hl7-org:v3")

# Per-file changelogs stored here
all_changelogs = {}   # filename → list of change dicts
file_summaries = []   # for the Summary sheet

for filepath in input_files:
    fname = os.path.basename(filepath)
    print(f"──── {fname} ────")

    # Parse XML (strip browser preamble if present)
    with open(filepath) as f:
        raw = f.read()
    raw = strip_preamble(raw)
    try:
        tree = ET.parse(__import__("io").StringIO(raw))
    except ET.ParseError as e:
        print(f"  PARSE ERROR: {e} — skipping")
        continue
    root = tree.getroot()

    # Detect namespace
    ns_m = re.match(r'\{(.+?)\}', root.tag)
    ns = ns_m.group(1) if ns_m else ""

    changelog = []

    # Pass 1 — structural rules
    p1 = pass1_structural(root, ns, changelog)
    print(f"  Pass 1 (structural): {p1} changes")

    # Pass 2 — ZeroShot NER on all text nodes
    p2 = pass2_nlp(root, changelog)
    print(f"  Pass 2 (NLP):        {p2} nodes updated")

    total = len(changelog)
    print(f"  Total logged:        {total}")

    # Save de-identified XML
    out_path = os.path.join(OUTPUT_DIR, fname)
    tree.write(out_path, encoding="unicode", xml_declaration=True)
    print(f"  Saved → {out_path}")
    print()

    all_changelogs[fname] = changelog
    file_summaries.append({
        "File"           : fname,
        "Pass 1 (structural)": p1,
        "Pass 2 (NLP)"   : p2,
        "Total Changes"  : total,
        "Output File"    : os.path.join("deid", fname),
    })

print(f"Batch complete. {len(all_changelogs)} files processed.")


## Excel Report — One Sheet Per File

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADERS    = ["#","XML Location","Field Type","Original Value","Rule Applied","De-identified Value"]
COL_WIDTHS = [4,   32,            18,           30,              22,            30]

header_fill = PatternFill("solid", fgColor="1F4E79")
header_font = Font(bold=True, color="FFFFFF", size=11)
alt_fill    = PatternFill("solid", fgColor="DEEAF1")
orig_fill   = PatternFill("solid", fgColor="FCE4D6")   # orange — original value
deid_fill   = PatternFill("solid", fgColor="E2EFDA")   # green  — de-identified
wrap_align  = Alignment(wrap_text=True, vertical="top")
thin_border = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

wb = openpyxl.Workbook()

# ── Summary sheet (first tab) ─────────────────────────────────────────────
ws_sum = wb.active
ws_sum.title = "Summary"

sum_headers = ["File","Pass 1 (structural)","Pass 2 (NLP)","Total Changes","Output File"]
sum_widths  = [20,     22,                   14,             16,             35]
sum_fill    = PatternFill("solid", fgColor="1F4E79")

for ci, (h, w) in enumerate(zip(sum_headers, sum_widths), 1):
    cell = ws_sum.cell(row=1, column=ci, value=h)
    cell.font = header_font; cell.fill = sum_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = thin_border
    ws_sum.column_dimensions[get_column_letter(ci)].width = w
ws_sum.row_dimensions[1].height = 22

total_all = 0
for ri, row in enumerate(file_summaries, 2):
    is_alt = ri % 2 == 0
    for ci, key in enumerate(sum_headers, 1):
        cell = ws_sum.cell(row=ri, column=ci, value=row[key])
        cell.alignment = Alignment(horizontal="center" if ci != 5 else "left", vertical="center")
        cell.border = thin_border
        if is_alt: cell.fill = alt_fill
    total_all += row["Total Changes"]

# Totals row
tr = len(file_summaries) + 2
ws_sum.cell(row=tr, column=1, value="TOTAL").font = Font(bold=True)
ws_sum.cell(row=tr, column=4, value=total_all).font = Font(bold=True)
ws_sum.freeze_panes = "A2"

# ── One sheet per file ────────────────────────────────────────────────────
for fname, changelog in all_changelogs.items():
    sheet_name = fname.replace(".txt","")   # e.g. "file1"
    ws = wb.create_sheet(title=sheet_name)

    # Header row
    for ci, (h, w) in enumerate(zip(HEADERS, COL_WIDTHS), 1):
        cell = ws.cell(row=1, column=ci, value=h)
        cell.font = header_font; cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[1].height = 22

    # Data rows
    for ri, entry in enumerate(changelog, 2):
        is_alt = ri % 2 == 0
        vals = [ri-1,
                entry["XML Location"], entry["Field Type"],
                entry["Original Value"], entry["Rule Applied"],
                entry["De-identified Value"]]
        for ci, val in enumerate(vals, 1):
            cell = ws.cell(row=ri, column=ci, value=str(val) if val else "")
            cell.alignment = Alignment(horizontal="center", vertical="top") if ci==1 else wrap_align
            cell.border = thin_border
            if   ci == 4: cell.fill = orig_fill
            elif ci == 6: cell.fill = deid_fill
            elif is_alt:  cell.fill = alt_fill
        ws.row_dimensions[ri].height = 40

    ws.freeze_panes = "A2"
    print(f"  Sheet '{sheet_name}': {len(changelog)} rows")

wb.save(EXCEL_OUT)
print(f"\nExcel saved: {EXCEL_OUT}")
print(f"Total changes across all files: {total_all}")


In [ ]:
spark.stop()
print("Spark stopped — license released.")
